In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from config import CONFIG

engine = create_engine(CONFIG["db_url"])
print("Connected")

Connected


In [2]:
df = pd.read_sql(
    f'SELECT * FROM "{CONFIG["schema"]}"."{CONFIG["clean_table"]}"',
    engine
)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded: 101,766 rows x 69 columns


In [3]:
# feature_engineering.py
import pandas as pd
import numpy as np
from config import CONFIG

def engineer_features(df):
    df_out = df.copy()

    # Medication column names after snake_case rename in Step 10
    med_cols = [c.replace("-", "_") for c in CONFIG["medication_cols"]]
    med_cols = [c for c in med_cols if c in df_out.columns]

    # Feature 1: Number of active (prescribed) medications
    # Counts medication columns where the value is NOT "No"
    df_out["num_active_medications"] = (
        df_out[med_cols]
        .apply(lambda row: (row != "No").sum(), axis=1)
    )

    # Feature 2: Any medication dose change during this encounter
    # 1 if any medication was titrated Up or Down
    df_out["any_medication_change"] = (
        df_out[med_cols]
        .apply(lambda row: int(row.isin(["Up", "Down"]).any()), axis=1)
    )

    # Feature 3: Numeric age midpoint from age band string
    # "[70-80)" becomes 75.0 — used in correlation analysis
    age_extracted = (
        df_out["age"]
        .str.extract(r'\[(\d+)-(\d+)\)')
        .astype(float)
    )
    df_out["age_midpoint"] = age_extracted.mean(axis=1)

    # Feature 4: Total prior healthcare utilization
    # Sum of outpatient + emergency + inpatient visits before this encounter
    df_out["total_prior_visits"] = (
        df_out["number_outpatient"] +
        df_out["number_emergency"]  +
        df_out["number_inpatient"]
    )

    # Feature 5: Polypharmacy flag
    # Clinical threshold: >= 10 medications is considered polypharmacy
    df_out["polypharmacy_flag"] = (df_out["num_medications"] >= 10).astype(int)

    # Summary
    print("--- Engineered Features Summary ---")
    for col in ["num_active_medications", "any_medication_change",
                "age_midpoint", "total_prior_visits", "polypharmacy_flag"]:
        print(f"{col:<28}: "
              f"min={df_out[col].min():.0f}  "
              f"max={df_out[col].max():.0f}  "
              f"mean={df_out[col].mean():.2f}")

    return df_out


In [4]:
df = engineer_features(df)

--- Engineered Features Summary ---
num_active_medications      : min=0  max=6  mean=1.18
any_medication_change       : min=0  max=1  mean=0.27
age_midpoint                : min=5  max=95  mean=65.97
total_prior_visits          : min=0  max=80  mean=1.20
polypharmacy_flag           : min=0  max=1  mean=0.80


##  Clinical Feature Engineering — Findings

**Purpose:**
Create five derived features from existing columns before export to
PostgreSQL. These features are computed here in Python because they
require row-level calculations across multiple columns — Python's
natural strength. SQL will use these features directly in EDA queries
and KPI views without needing to recalculate them.

**Features Created:**

**Feature 1: num_active_medications**
- Definition: Count of medication columns where value is NOT "No"
  — represents medications actively prescribed during the encounter
- Range: 0 to 6 | Mean: 1.18
- Interpretation: On average patients have 1.18 active diabetes
  medications prescribed per encounter. The maximum of 6 means
  some patients are on 6 different diabetes medications simultaneously.
  This is distinct from num_medications (total medications including
  non-diabetes drugs) which has a mean of 16.02 — confirming that
  most medications prescribed are for comorbid conditions rather
  than diabetes management specifically.

**Feature 2: any_medication_change**
- Definition: Binary flag — 1 if any medication dose was titrated
  Up or Down during the encounter, 0 if all doses remained Steady
  or were not prescribed
- Range: 0 to 1 | Mean: 0.27
- Interpretation: 27% of encounters involved at least one medication
  dose adjustment during the hospital stay. This is a clinical
  indicator of active diabetes management — the EDA phase will
  test whether medication adjustment correlates with lower
  readmission rates, which would suggest that proactive management
  during admission reduces the risk of returning.

**Feature 3: age_midpoint**
- Definition: Numeric midpoint extracted from age band string
  e.g. "[70-80)" becomes 75.0
- Range: 5 to 95 | Mean: 65.97
- Interpretation: The mean age midpoint of 65.97 years confirms
  the dataset is heavily skewed toward older adults, consistent
  with the age band distribution observed in Step 4 where [70-80)
  was the largest group at 25.62%. The minimum of 5 corresponds
  to the [0-10) band and the maximum of 95 corresponds to [90-100).
  This numeric representation enables correlation analysis between
  age and readmission risk, which the string age band format
  cannot support.

**Feature 4: total_prior_visits**
- Definition: Sum of number_outpatient + number_emergency +
  number_inpatient — total prior healthcare contacts in the
  year before this encounter
- Range: 0 to 80 | Mean: 1.20
- Interpretation: On average patients had 1.20 prior healthcare
  contacts in the year before this encounter. The maximum of 80
  represents an extreme high-utilization patient — consistent with
  the maximum of 76 emergency visits seen in the numeric profile
  in Step 5. This composite feature is more analytically useful
  than any individual utilization column because it captures
  total healthcare burden in a single value. It will be a key
  input into the high-risk segment KPI view.

**Feature 5: polypharmacy_flag**
- Definition: Binary flag — 1 if num_medications >= 10
  (clinical threshold for polypharmacy), 0 otherwise
- Range: 0 to 1 | Mean: 0.80
- Interpretation: 80% of encounters involve patients on 10 or
  more medications — a striking finding. This very high rate
  is clinically consistent with a diabetes inpatient population
  where patients typically present with multiple comorbidities
  (hypertension, cardiovascular disease, kidney disease) each
  requiring their own medication regimen alongside diabetes
  management. The polypharmacy_flag will be used in the KPI
  views to compare readmission rates between polypharmacy and
  standard medication burden patients.

**Feature Summary Table:**

| Feature | Min | Max | Mean | Type |
|---|---|---|---|---|
| num_active_medications | 0 | 6 | 1.18 | Count |
| any_medication_change | 0 | 1 | 0.27 | Binary flag |
| age_midpoint | 5 | 95 | 65.97 | Continuous numeric |
| total_prior_visits | 0 | 80 | 1.20 | Count |
| polypharmacy_flag | 0 | 1 | 0.80 | Binary flag |

**Notable Finding — Polypharmacy Rate:**
The polypharmacy mean of 0.80 means 80% of all encounters in this
dataset involve patients on 10 or more medications. This is one of
the most striking descriptive findings in the entire project and
will be prominently featured in the dashboard. It contextualizes
why this patient population has high readmission risk — they are
elderly, diabetic, and managing multiple chronic conditions
simultaneously.

**Why These Features Were Created in Python Not SQL:**
All five features require row-level calculations across multiple
columns simultaneously — counting non-No values across 23 medication
columns, extracting numbers from age band strings, summing three
utilization columns. These operations are natural in pandas but
would require complex subqueries or repeated CASE statements in SQL.
Creating them here means every SQL KPI view can reference a simple
column name rather than re-deriving the calculation each time.

**Next Step:** Pre-export validation — verify all cleaning and
engineering steps produced expected outputs before writing to
PostgreSQL.

In [5]:
# pre_export_validation.py

def validate_before_export(df, sentinel="?"):
    print("=== Pre-Export Validation ===\n")

    print(f"Shape              : {df.shape[0]:,} rows  x  {df.shape[1]} columns")

    sentinel_remaining = (df == sentinel).sum().sum()
    print(f"Sentinel values    : {sentinel_remaining}  (expected 0)")

    pk_nulls = df[CONFIG["primary_key"]].isna().sum()
    print(f"Primary key nulls  : {pk_nulls}  (expected 0)")

    flag_cols = [c for c in df.columns if c.endswith("_flag")]
    for col in flag_cols:
        unique_vals = sorted(df[col].unique())
        non_binary  = [v for v in unique_vals if v not in [0, 1]]
        print(f"  {col:<35}: unique={unique_vals}  non-binary={len(non_binary)}")

    expected_features = [
        "num_active_medications", "any_medication_change",
        "age_midpoint", "total_prior_visits", "polypharmacy_flag",
        "weight_available"
    ]
    for feat in expected_features:
        status = "OK" if feat in df.columns else "MISSING"
        print(f"  {feat:<35}: {status}")

    unexpected_target = df[~df[CONFIG["target_col"]]
                           .isin(CONFIG["valid_readmitted"])][CONFIG["target_col"]].unique()
    print(f"Unexpected target values: {unexpected_target}  (expected empty)")

    print("\nValidation complete.")

validate_before_export(df)

=== Pre-Export Validation ===

Shape              : 101,766 rows  x  69 columns
Sentinel values    : 0  (expected 0)
Primary key nulls  : 0  (expected 0)
  duplicate_flag                     : unique=[np.int64(0)]  non-binary=0
  diag_1_missing_flag                : unique=[np.int64(0), np.int64(1)]  non-binary=0
  gender_invalid_flag                : unique=[np.int64(0), np.int64(1)]  non-binary=0
  expired_or_hospice_flag            : unique=[np.int64(0), np.int64(1)]  non-binary=0
  time_in_hospital_outlier_flag      : unique=[np.int64(0), np.int64(1)]  non-binary=0
  num_lab_procedures_outlier_flag    : unique=[np.int64(0), np.int64(1)]  non-binary=0
  num_procedures_outlier_flag        : unique=[np.int64(0), np.int64(1)]  non-binary=0
  num_medications_outlier_flag       : unique=[np.int64(0), np.int64(1)]  non-binary=0
  number_outpatient_outlier_flag     : unique=[np.int64(0), np.int64(1)]  non-binary=0
  number_emergency_outlier_flag      : unique=[np.int64(0), np.int64(1)]  no

In [6]:
import pandas as pd
from sqlalchemy import create_engine, text
from config import CONFIG


def safe_export_clean_table(df, engine, schema, table):
    """Safely updates PostgreSQL data without breaking views or Power BI DirectQuery."""
    # engine.begin() creates an atomic transaction block.
    # If anything fails during insert, it rolls back automatically so your DB is never corrupted.
    with engine.begin() as conn:
        # 1. Clear existing rows without dropping table or views
        conn.execute(
            text(f'TRUNCATE TABLE "{schema}"."{table}" RESTART IDENTITY;')
        )

        # 2. Append updated dataframe
        df.to_sql(
            name=table,
            con=conn,
            schema=schema,
            if_exists="append",  # Crucial: appends rows, does NOT drop table
            index=False,
            method="multi",
            chunksize=1000,
        )

    # 3. Verification query
    count = pd.read_sql(
        f'SELECT COUNT(*) FROM "{schema}"."{table}"', engine
    ).iloc[0, 0]
    print(
        f"✅ Export Successful! Python rows: {len(df):,} | PostgreSQL confirmed: {count:,}"
    )


# Run export
engine = create_engine(CONFIG["db_url"])
safe_export_clean_table(df, engine, CONFIG["schema"], CONFIG["clean_table"])

✅ Export Successful! Python rows: 101,766 | PostgreSQL confirmed: 101,766
